In [ ]:

import resource
resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY)) 

In [ ]:
import os  

def rename_fch_to_fchk(directory: str = ".") -> None:
    
    
    directory = os.path.abspath(directory)  
    print(f"Checking directory: {directory}")  

    
    for name in os.listdir(directory):  
        old_path = os.path.join(directory, name)  

        
        if not os.path.isfile(old_path):  
            continue  

        
        if name.endswith(".fch"):  
            
            new_name = name[:-4] + ".fchk"  
            new_path = os.path.join(directory, new_name)  

            
            if os.path.exists(new_path):  
                print(f"Skip: {old_path} -> {new_path} (target already exists)")  
                continue  

            
            os.rename(old_path, new_path)  
            print(f"Renamed: {old_path} -> {new_path}")  

    print("Renaming finished.")  


rename_fch_to_fchk(".")

In [ ]:
import os                       
import re                       
import subprocess               
import signal                   
import tempfile                 
import textwrap                 
from pathlib import Path        
from typing import List, Tuple, Optional
from shutil import which        

def get_fchk_charge(fchk_path: os.PathLike) -> Optional[int]:
    
    p = Path(fchk_path)
    if not p.is_file():
        return None
    try:
        with open(p, "r", encoding="utf-8", errors="ignore") as fh:
            lines = fh.readlines()
        charge_idx = None
        for i, line in enumerate(lines):
            if re.match(r"^\s*Charge\b", line, flags=re.IGNORECASE):
                charge_idx = i
                break
        if charge_idx is None:
            
            for i, line in enumerate(lines):
                if re.search(r"\bNet\s*Charge\b", line, flags=re.IGNORECASE):
                    charge_idx = i
                    break
        if charge_idx is None:
            return None

        
        for j in range(charge_idx, min(charge_idx + 5, len(lines))):
            nums = re.findall(r"(-?\d+)", lines[j])
            if nums:
                return int(nums[0])
        return None
    except Exception:
        return None

In [ ]:



def calculate_minmax_ESP(fchk_folder_path: str, filename: str) -> Tuple[float, float, float, float]:
    
    folder = Path(fchk_folder_path).expanduser().resolve()
    fchk_file = folder / f"{os.path.splitext(os.path.basename(filename))[0]}.fchk"
    if not fchk_file.is_file():
        raise FileNotFoundError("Public message removed for release.")

    cmd = ["Multiwfn", str(fchk_file)]
    seq = "\n".join(["12", "0", "-1", "-1", "q"]) + "\n"

    proc = subprocess.run(
        cmd, input=seq, text=True, cwd=folder,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False
    )
    out = proc.stdout or ""

    
    ESP_min_au: Optional[float] = None
    ESP_min_eV: Optional[float] = None
    ESP_max_au: Optional[float] = None
    ESP_max_eV: Optional[float] = None

    section = None  
    
    star_line_re = re.compile(
        r"^\*\s*\d+\s+([\-+]?\d+(?:\.\d+)?(?:[EeDd][\-+]?\d+)?)\s+([\-+]?\d+(?:\.\d+)?(?:[EeDd][\-+]?\d+)?)"
    )

    for raw in out.splitlines():
        line = raw.strip("\n")
        if "The number of surface minima" in line:
            section = "min"
            continue
        if "The number of surface maxima" in line:
            section = "max"
            continue

        if section in ("min", "max") and line.lstrip().startswith("*"):
            m = star_line_re.match(line)
            if m:
                au = float(m.group(1).replace("D", "E"))
                ev = float(m.group(2).replace("D", "E"))
                if section == "min":
                    ESP_min_au, ESP_min_eV = au, ev
                else:
                    ESP_max_au, ESP_max_eV = au, ev
                
                section = None

    if None in (ESP_min_au, ESP_min_eV, ESP_max_au, ESP_max_eV):
        raise RuntimeError("Public message removed for release.")

    return ESP_min_au, ESP_min_eV, ESP_max_au, ESP_max_eV


In [ ]:
import os                       
import subprocess               
import signal                   
from pathlib import Path        
from typing import List         

def batch_generate_esp_cub(fchk_folder_path: str) -> List[Path]:
    
    
    input_dir = Path(fchk_folder_path).expanduser().resolve()

    
    if not input_dir.is_dir():
        raise NotADirectoryError("Public message removed for release.")

    
    cmd_sequence = "\n".join(["5", "1", "3", "2", "0", "5", "12", "1", "2", "0", "q"])

    
    filename_list: List[Path] = []

    
    for fchk_path in input_dir.glob("*.fchk"):
        
        filename = fchk_path.stem
        print(filename)

        
        density_tmp = input_dir / "density.cub"
        esp_tmp = input_dir / "totesp.cub"

        
        if density_tmp.exists():
            density_tmp.unlink()
        if esp_tmp.exists():
            esp_tmp.unlink()

        
        cmd = ["Multiwfn", str(fchk_path), "-ESPrhoiso", "0.001"]

        
        with subprocess.Popen(
            cmd,
            stdin=subprocess.PIPE,      
            stdout=subprocess.PIPE,     
            stderr=subprocess.PIPE,
            text=True,                  
            cwd=input_dir               
        ) as proc:
            
            proc.communicate(input=cmd_sequence)

        
        density_new = input_dir / f"{filename}_density.cub"
        esp_new = input_dir / f"{filename}_ESP.cub"

        
        if density_tmp.exists():
            density_tmp.rename(density_new)
        else:
            print("Public message removed for release.")

        if esp_tmp.exists():
            esp_tmp.rename(esp_new)
        else:
            print("Public message removed for release.")
        
        
        filename_list.append(filename)
    
    return filename_list

In [ ]:
import os  

def generate_esp_vmd(fchk_folder_path, filename: str) -> None:
    
    
    
    
    basename = os.path.splitext(os.path.basename(filename))[0]  
    fchk_path = Path(fchk_folder_path).expanduser().resolve() / f"{basename}.fchk"

    
    colorlow_au  = -0.03
    colorhigh_au =  0.03
    
    colorlow_ev_comm  = -0.8
    colorhigh_ev_comm =  0.8

    
    charge = get_fchk_charge(fchk_path)
    if charge is not None and charge != 0:
        try:
            min_au, min_ev, max_au, max_ev = calculate_minmax_ESP(fchk_folder_path, basename)
            
            lo_au, hi_au = sorted([min_au, max_au])
            colorlow_au, colorhigh_au = lo_au, hi_au
            
            lo_ev, hi_ev = sorted([min_ev, max_ev])
            colorlow_ev_comm, colorhigh_ev_comm = lo_ev, hi_ev
            print(f"[INFO] {basename}: charge={charge}, "
                  f"ESP range (a.u.)=({colorlow_au:.8f}, {colorhigh_au:.8f}); "
                  f"(eV)=({colorlow_ev_comm:.6f}, {colorhigh_ev_comm:.6f})")
        except Exception as e:
            print("Public message removed for release.")

    
    
    vmd_script = f"""#This script is used to draw ESP colored molecular vdW surface (rho=0.001)

color scale method BWR
color Display Background white
axes location Off
display depthcue off
display rendermode GLSL
light 2 on
light 3 on
material change transmode EdgyGlass 1.0
material change specular EdgyGlass 0.15
material change shininess EdgyGlass 0.95
material change opacity EdgyGlass 0.7
material change outlinewidth EdgyGlass 0.9
material change outline EdgyGlass 0.5

#The maximum number of systems to be loaded
set nsystem 1
#Lower and upper limit of color scale of ESP (a.u.)
set colorlow {colorlow_au:.8f}
set colorhigh {colorhigh_au:.8f}
# eV as unit (for reference)
# set colorlow  {colorlow_ev_comm:.6f}
# set colorhigh {colorhigh_ev_comm:.6f}

for {{set i 1}} {{${{i}}<=$nsystem}} {{incr i}} {{
set id [expr $i-1]
mol new {fchk_folder_path}/{basename}_density.cub
mol addfile {fchk_folder_path}/{basename}_ESP.cub
mol modstyle 0 $id CPK 1.000000 0.300000 22.000000 22.000000
mol addrep $id
mol modstyle 1 $id Isosurface 0.001000 0 0 0 1 1
mol modmaterial 1 $id EdgyGlass
mol modcolor 1 $id Volume 1
mol scaleminmax $id 1 $colorlow $colorhigh
}}
"""
    
    output_name = f"{fchk_folder_path}/{basename}_ESPiso.vmd"  

    
    with open(output_name, "w", encoding="utf-8") as f:  
        f.write(vmd_script)  
    
    print(f"VMD script saved to: {os.path.abspath(output_name)}")  

In [ ]:
from pathlib import Path
import os

def generate_colorbar_vmd(fchk_folder_path: str, filename: str) -> None:
    
    basename = os.path.splitext(os.path.basename(filename))[0]
    fchk_path = Path(fchk_folder_path).expanduser().resolve() / f"{basename}.fchk"

    
    colorlow_au, colorhigh_au = -0.03, 0.03
    charge = get_fchk_charge(fchk_path)
    if charge is not None and charge != 0:
        try:
            min_au, _min_ev, max_au, _max_ev = calculate_minmax_ESP(fchk_folder_path, basename)
            colorlow_au, colorhigh_au = sorted([min_au, max_au])
        except Exception as e:
            print("Public message removed for release.")

    
    vmd_script = 
    out_path = Path(fchk_folder_path) / f"{basename}_COLORBAR.vmd"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(vmd_script)
    print(f"Color bar VMD script saved to: {os.path.abspath(out_path)}")

In [ ]:




import os               
import subprocess        
import tempfile          
import textwrap          
from shutil import which 

def render_esp_with_vmd(fchk_folder_path, filename: str) -> None:
    
    
    vmd_cmd = which('vmd')                         
    if vmd_cmd is None:                           
        raise FileNotFoundError(
            "Public message removed for release."
        )
    
    vmd_script = Path(fchk_folder_path, f"{filename}_ESPiso.vmd").resolve()
    output_img = Path(fchk_folder_path, f"{filename}_ESP.tga").resolve()

    if not os.path.isfile(vmd_script):             
        raise FileNotFoundError(
            "Public message removed for release."
        )
    
    
    
    tcl_commands = textwrap.dedent().strip()                                  
    
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='.tcl', delete=False
    ) as tcl_file:                                
        tcl_file.write(tcl_commands)              
        temp_tcl_path = tcl_file.name             
    
    
    
    try:
        result = subprocess.run(
            [vmd_cmd, '-dispdev', 'text', '-e', temp_tcl_path],
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            cwd=vmd_script.parent,     
            env={**os.environ, "VMDNOCUDA": "1"}  
        )
    finally:
        os.remove(temp_tcl_path)                  
    
    
    if result.returncode != 0 or (not output_img.exists()) or os.path.getsize(output_img) == 0:
        err_msg = (
            "Public message removed for release."
            "Public message removed for release."
            f"--- stdout ---\n{result.stdout}\n"
            f"--- stderr ---\n{result.stderr}"
        )
        raise RuntimeError(err_msg)
    else:
        print("Public message removed for release.")  

In [ ]:
def render_colorbar_with_vmd(fchk_folder_path: str, filename: str) -> None:
    
    vmd_cmd = which('vmd')
    if vmd_cmd is None:
        raise FileNotFoundError("Public message removed for release.")

    basename = os.path.splitext(os.path.basename(filename))[0]
    vmd_script = f"{fchk_folder_path}/{basename}_COLORBAR.vmd"
    output_img = f"{fchk_folder_path}/{basename}_color_bar.tga"

    if not os.path.isfile(vmd_script):
        raise FileNotFoundError("Public message removed for release.")

    
    tcl_commands = textwrap.dedent(f"""
        source "{vmd_script}"
        render TachyonInternal "{output_img}"
        quit
    """).strip()

    with tempfile.NamedTemporaryFile(mode='w', suffix='.tcl', delete=False) as tcl_file:
        tcl_file.write(tcl_commands)
        temp_tcl_path = tcl_file.name

    try:
        result = subprocess.run(
            [vmd_cmd, '-dispdev', 'text', '-e', temp_tcl_path],
            check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
    finally:
        os.remove(temp_tcl_path)

    if result.returncode != 0:
        err_msg = (
            "Public message removed for release."
            f"--- stdout ---\n{result.stdout}\n"
            f"--- stderr ---\n{result.stderr}"
        )
        raise RuntimeError(err_msg)
    else:
        print("Public message removed for release.")

In [ ]:
filename_list = batch_generate_esp_cub(".")
for filename in filename_list:
    
    generate_esp_vmd(".", filename)
    render_esp_with_vmd(".", filename)
    
    generate_colorbar_vmd(".", filename)
    render_colorbar_with_vmd(".", filename)